> 📅 __Date: 2026-08-05__

# **Attention Mechanism**

The **Attention Mechanism** was introduced in **2015** to overcome the major limitation of the Encoder-Decoder architecture.

In the Encoder-Decoder model, the entire input sequence is compressed into a single **Context Vector**.

For long sentences, this fixed-length vector often loses important information, resulting in poor translations.

The Attention Mechanism allows the decoder to **focus on the most relevant parts of the input sequence** while generating each output word.

---

# 🚀 **Why Attention Was Introduced**

Consider the sentence

```text
I like to eat apple.
```

Here, **apple** refers to a **fruit**.

Now consider another sentence

```text
Apple launched a new phone.
```

Here, **Apple** refers to the **company**.

The meaning of a word depends on its surrounding words (context).

Traditional embedding methods could not capture this dynamic meaning.

---

# 📦 **Traditional Embedding Methods (Static Embeddings)**

Methods such as

- Word2Vec
- GloVe
- FastText

generate **one fixed vector** for every word.

For example,

```text
Apple

↓

Embedding

↓

[0.32, -0.15, 0.81, ...]
```

Regardless of the sentence,

```text
I like to eat apple.
```

and

```text
Apple launched a new phone.
```

the word **apple** receives the **same embedding vector**.

Therefore, these methods are called **Static Embeddings**.

### **Limitation**

- Same vector for every occurrence of a word.
- Cannot capture different meanings based on context.
- Polysemous words (words with multiple meanings) are not represented well.

This motivated the development of the **Attention Mechanism**, which creates **context-aware representations**.

---

# 🧠 **Basic Idea of Attention**

Instead of treating every word equally, Attention assigns **different importance (weights)** to different words.

While processing a word, the model asks:

> **Which other words are most relevant to understanding this word?**

The more relevant a word is, the more attention it receives.

---

# **Scaled Dot-Product Attention**

Scaled Dot-Product Attention is the fundamental building block of the Transformer architecture.

It consists of four major steps.

---

# 🌍 **Example**

Suppose we have the sentence

```text
Apple launched new phone.
```

---

# **Step 1 — Tokenize & Vectorize**

The sentence is first tokenized.

```text
["Apple", "launched", "new", "phone"]
```

Each token is converted into an embedding vector.

```text
Apple     → A

launched  → L

new        → N

phone      → P
```

These embedding vectors are then transformed into three different vectors.

- Query (Q)
- Key (K)
- Value (V)

```text
A, L, N, P

↓

Linear Layers

↓

Query (Q)

Key (K)

Value (V)
```

---

# 🧠 **Query, Key and Value Vectors**

Every input token is converted into three vectors.

```text
Apple

↓

Query   → Aq

Key     → Ak

Value   → Av
```

Similarly,

```text
launched

↓

Lq

Lk

Lv
```

```text
new

↓

Nq

Nk

Nv
```

```text
phone

↓

Pq

Pk

Pv
```

---

# 📌 **Meaning of Query, Key and Value**

### **Query (Q)**

The Query represents the word whose meaning we want to understand.

Example

```text
Apple
```

We ask:

> Which words in the sentence are important for understanding **Apple**?

---

### **Key (K)**

The Key represents the information that each word can provide.

Every Query is compared with all the Keys to determine how relevant each word is.

---

### **Value (V)**

The Value contains the actual information that will be combined to create the final contextual representation.

---

### **Simple Intuition**

- **Query (Q)** → Which words are important for me?
- **Key (K)** → What information does each word contain?
- **Value (V)** → What information will actually be used to build my new representation?

---

# 🤔 **Why do we need Query, Key and Value?**

Suppose we want to understand the word **Apple** in the sentence

```text
Apple launched a new phone.
```

The model works as follows:

- The **Query (Apple)** asks:

  > Which words in this sentence are important for understanding me?

- Every **Key** describes the information available in each word.

- The Query is compared with every Key to measure how relevant each word is.

- These relevance scores are converted into attention probabilities using the **Softmax** function.

- Finally, the attention probabilities are used to compute a weighted sum of the **Value vectors**, producing the contextual representation of **Apple**.

In simple terms,

- **Query** asks the question.
- **Key** determines which words are relevant.
- **Value** provides the information used to build the final contextual representation.

---

# **Step 2 — Compute Similarity Scores**

To determine which words are important, we compute the **Dot Product** between the Query and every Key.

For the word **Apple**,

```text
Aq · Ak → S11

Aq · Lk → S12

Aq · Nk → S13

Aq · Pk → S14
```

These values are called **Attention Scores**.

---

# 📐 **Dot Product**

The dot product is

$$
a · b = |a||b|cos(θ)
$$

### **Observation**

- Larger value → Higher similarity
- Smaller value → Lower similarity

Therefore,

> The larger the dot product, the more relevant that word is.

---

# **Step 3 — Scaling**

If the Query and Key vectors have a high dimension, their dot-product values can become very large.

Large attention scores cause the Softmax function to produce extremely peaked probabilities, making training unstable and resulting in very small gradients.

To avoid this problem, every attention score is divided by the square root of the Key vector dimension.

$$
\sqrt{d_k}
$$

where

- $d_k$ = Dimension of the Key vectors.

Therefore, the scaled attention score is

$$
\text{Scaled Score}
=
\frac{QK^\top}{\sqrt{d_k}}
$$

---

For the word **Apple**, let its Query vector be $A_q$.

The scaled attention scores are

$$
S_{11}
=
\frac{A_q \cdot A_k}{\sqrt{d_k}}
$$

$$
S_{12}
=
\frac{A_q \cdot L_k}{\sqrt{d_k}}
$$

$$
S_{13}
=
\frac{A_q \cdot N_k}{\sqrt{d_k}}
$$

$$
S_{14}
=
\frac{A_q \cdot P_k}{\sqrt{d_k}}
$$

---

# **Step 4 — Softmax**

The scaled scores are converted into probabilities using the **Softmax** function.

```text
Softmax

↓

P11

P12

P13

P14
```

These probabilities tell us **how much attention** Apple should pay to every word.

---

## **PyTorch Example**

```python
import torch
from torch.nn.functional import softmax

logits = torch.tensor([105.0, 98.0, 97.0, 92.0])

softmax(logits, dim=0)
```

Output

```python
tensor([9.9875e-01, 9.1074e-04, 3.3504e-04, 2.2575e-06])
```

---

# 📌 **Observation**

### **Dot Product**

- Larger score → More similar

### **Softmax**

- Larger score → Higher probability

### **Conclusion**

> More similar the vectors, higher the attention weight.

---

# **Step 5 — Weighted Sum**

The attention probabilities are now used to compute a weighted sum of the **Value vectors**.

For the word **Apple**, the output vector is

$$
V_1
=
P_{11}A_v
+
P_{12}L_v
+
P_{13}N_v
+
P_{14}P_v
$$

where

- $P_{11}, P_{12}, P_{13}, P_{14}$ are the attention probabilities obtained from the Softmax function.
- $A_v, L_v, N_v, P_v$ are the corresponding **Value vectors**.

The resulting vector $V_1$ is called the **Contextual Representation** (or **Attention Output**) of the word **Apple**.

Unlike static embeddings, this representation depends on the surrounding words in the sentence.

---

In matrix form, this operation is written as

$$
\text{Attention Output}
=
\text{Softmax}
\left(
\frac{QK^\top}{\sqrt{d_k}}
\right)
V
$$

This is the complete equation for **Scaled Dot-Product Attention**.

---

# 🖼️ **Self-Attention Architecture**

<div align="center">

<img src="assets/self-attention.png" width="300" alt="Self Attention">

<p><em>Figure: Self-Attention mechanism.</em></p>

</div>

---

# 🧠 **Contextual Embedding**

Initially,

```text
Apple

↓

Embedding

↓

A
```

After Self-Attention,

```text
Apple

↓

Contextual Representation

↓

V₁
```

The representation now depends on the entire sentence.

---

# 🔄 **Self-Attention**

In Self-Attention, the same input sequence is used to compute

- Query
- Key
- Value

Example

```text
Apple launched new phone.
```

The model allows every word to attend to every other word in the same sentence.

---

# 🔀 **Cross-Attention**

Cross-Attention uses **two different sequences**.

- Query comes from one sequence.
- Key and Value come from another sequence.

Example

```text
English

Apple launched new phone.
```

↓

```text
Hindi

एप्पल ने नया फोन लॉन्च किया।
```

Cross-Attention is widely used in

- Machine Translation
- Encoder-Decoder Transformers

---

# 🔄 **Types of Attention**

## **1. Self-Attention**

- Same sequence is used for Query, Key and Value.
- Used inside the Transformer Encoder.

---

## **2. Cross-Attention**

- Query comes from one sequence.
- Key and Value come from another sequence.
- Used in Encoder-Decoder Transformers.

---

# 🧠 **Why Multi-Head Attention?**

Consider the sentence

```text
Tim Cook is CEO of Apple. He is a great person.
```

To understand this sentence, the model needs to capture

- Meaning
- Syntax
- Grammar
- References (e.g., "He" refers to Tim Cook)
- Long-range dependencies

A single attention calculation may not capture all these relationships effectively.

---

# **Multi-Head Attention**

Instead of computing attention only once, the Transformer computes attention **multiple times in parallel**.

Example

```text
4 Heads
```

Head 1

```text
Meaning
```

Head 2

```text
Syntax
```

Head 3

```text
Grammar
```

Head 4

```text
References
```

Each head learns to focus on different linguistic patterns.

---

# 🔄 **Processing in Multi-Head Attention**

Each attention head independently performs the same sequence of operations.

1. Tokenize & Vectorize
2. Generate Query, Key and Value
3. Compute Dot Product
4. Apply Scaling
5. Apply Softmax
6. Compute Weighted Sum

If there are **H heads**, attention is computed **H times in parallel**.

---

# ❓ **How Do Different Heads Learn Different Information?**

Although every head receives the same input sentence,

they use **different learnable weight matrices**.

For Head 1,

```text
Wq₁

Wk₁

Wv₁
```

For Head 2,

```text
Wq₂

Wk₂

Wv₂
```

Similarly, every head has its own

- Wq
- Wk
- Wv

Because these weights are different, every head learns different relationships.

For example,

- One head may learn grammar.
- Another may learn subject-object relationships.
- Another may learn long-distance dependencies.
- Another may learn semantic meaning.

---

# 📌 **Summary**

- Attention allows the model to focus on the most relevant words in a sentence.
- Traditional embeddings are static and assign one vector per word.
- Attention generates context-aware representations.
- Every token is converted into Query, Key and Value vectors.
- Query and Key are compared using the Dot Product.
- Scores are scaled by √dₖ.
- Softmax converts scores into attention probabilities.
- These probabilities are multiplied with the Value vectors to obtain contextual representations.
- Self-Attention uses the same sequence for Query, Key and Value.
- Cross-Attention uses different sequences.
- Multi-Head Attention computes attention multiple times in parallel using different learnable weight matrices.
- The Attention Mechanism solved the fixed Context Vector bottleneck of the Encoder-Decoder architecture and became the foundation of the Transformer.